In [0]:
# ============================================================
# NOTEBOOK: nb_04_Deduplication
# PURPOSE:  Applies the participant matching rules to
#           silver.participant_event and produces
#           silver.participant_event_dedup with a
#           participant_key and a reportable flag.
#
#           Also writes silver.excluded_records capturing
#           every row that is not reportable, with a reason.
#
# MATCHING TIERS:
#   Tier 1 - sa_id_normalised (13 digits, non-null)
#   Tier 2 - surname + first_name + persal_number
#   Tier 3 - event_key (each row is its own participant)
#
# REPORTING PERIOD:
#   Q4 2025 (2025-10-01 to 2025-12-31)
#
# CATALOG:  ktu_assessment_dev
# COMPUTE:  Serverless
# INPUT:    silver.participant_event
# OUTPUT:   silver.participant_event_dedup
#           silver.excluded_records
#           audit.pipeline_run_log
#           audit.data_quality_results
# ============================================================

import uuid
from datetime import datetime

from pyspark.sql import functions as F

CATALOG         = "ktu_assessment_dev"
AUDIT_SCHEMA    = "audit"
SILVER_SCHEMA   = "silver"

REPORTING_PERIOD_START = "2025-10-01"
REPORTING_PERIOD_END   = "2025-12-31"

DEBUG = 1

run_id     = str(uuid.uuid4())
notebook   = "nb_04_Deduplication"
start_time = datetime.now()

if DEBUG:
    print("=" * 50)
    print("NB_04_DEDUPLICATION STARTED")
    print("=" * 50)
    print(f"Run ID           : {run_id}")
    print(f"Reporting Period : {REPORTING_PERIOD_START} to {REPORTING_PERIOD_END}")
    print(f"Start Time       : {start_time}")

NB_04_DEDUPLICATION STARTED
Run ID           : 70d8c26b-7ae8-4505-a8c1-d9a367f1dde3
Reporting Period : 2025-10-01 to 2025-12-31
Start Time       : 2026-09-12 14:52:09.416087


In [0]:
# ============================================================
# AUDIT START AND TARGET TABLE CREATION
# ============================================================

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    VALUES (
        '{run_id}', '{notebook}', 'silver', 'silver.participant_event_dedup',
        '{start_time.strftime("%Y-%m-%d %H:%M:%S")}',
        NULL, 'RUNNING', 0, 0, 0, 'Deduplication in progress', 0
    )
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.excluded_records (
        event_key        STRING COMMENT 'Key from silver.participant_event',
        source_system    STRING COMMENT 'capturing_tool, chw, online_export',
        exclusion_reason STRING COMMENT 'Reason the row is not reportable',
        details          STRING COMMENT 'Free-text detail',
        excluded_at      TIMESTAMP
    )
    USING DELTA
""")

if DEBUG:
    print("Audit RUNNING row written and excluded_records table ready.")

Audit RUNNING row written and excluded_records table ready.


In [0]:
# ============================================================
# PARTICIPANT KEY CONSTRUCTION
# ============================================================
#
# WHAT THIS CELL DOES:
# Adds participant_key to every row using a three-tier rule.
#
# TIER 1 - SA ID:
# If sa_id_normalised is 13 digits and non-null, participant_key
# is the SA ID. This is the strongest deterministic identifier
# available. Luhn validity is NOT required because the
# assessment anonymised personal data; Luhn failure is
# expected and documented.
#
# TIER 2 - Surname + First name + PERSAL:
# When SA ID is missing but surname, first name, and PERSAL
# number are all present, participant_key is a hash of the
# three. This is a composite identifier with much lower
# collision risk than name alone.
#
# TIER 3 - event_key:
# When neither Tier 1 nor Tier 2 applies, participant_key
# falls back to the row's event_key. This treats the row
# as a distinct participant. Documented as a limitation:
# rows without a usable identifier cannot be linked across
# sources.
#
# IDENTIFIER PRIORITY FLAG:
# participant_match_tier records which tier was used, so the
# deduplication is auditable.
# ============================================================

fact = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.participant_event")

# Normalise name fields for matching
fact_norm = fact.withColumn(
    "surname_upper", F.upper(F.trim(F.coalesce(F.col("surname"), F.lit(""))))
).withColumn(
    "first_name_upper", F.upper(F.trim(F.coalesce(F.col("first_name"), F.lit(""))))
).withColumn(
    "persal_clean", F.trim(F.coalesce(F.col("persal_number"), F.lit("")))
)

# Build the three-tier key
fact_keyed = (
    fact_norm
    .withColumn(
        "tier1_key",
        F.when(
            (F.col("sa_id_normalised").isNotNull()) &
            (F.length(F.col("sa_id_normalised")) == 13),
            F.col("sa_id_normalised")
        ).otherwise(F.lit(None).cast("string"))
    )
    .withColumn(
        "tier2_key",
        F.when(
            (F.col("tier1_key").isNull()) &
            (F.length(F.col("surname_upper"))    > 0) &
            (F.length(F.col("first_name_upper")) > 0) &
            (F.length(F.col("persal_clean"))     > 0),
            F.sha2(F.concat_ws("||",
                F.col("surname_upper"),
                F.col("first_name_upper"),
                F.col("persal_clean")
            ), 256)
        ).otherwise(F.lit(None).cast("string"))
    )
    .withColumn(
        "participant_key",
        F.coalesce(
            F.col("tier1_key"),
            F.col("tier2_key"),
            F.col("event_key")
        )
    )
    .withColumn(
        "participant_match_tier",
        F.when(F.col("tier1_key").isNotNull(), F.lit("TIER_1_SA_ID"))
         .when(F.col("tier2_key").isNotNull(), F.lit("TIER_2_NAME_PERSAL"))
         .otherwise(F.lit("TIER_3_EVENT_KEY"))
    )
)

if DEBUG:
    print("=" * 60)
    print("PARTICIPANT KEY TIER DISTRIBUTION")
    print("=" * 60)
    fact_keyed.groupBy("participant_match_tier").agg(
        F.count("*").alias("rows"),
        F.countDistinct("participant_key").alias("distinct_participants")
    ).orderBy("participant_match_tier").show(truncate=False)

PARTICIPANT KEY TIER DISTRIBUTION
+----------------------+-----+---------------------+
|participant_match_tier|rows |distinct_participants|
+----------------------+-----+---------------------+
|TIER_1_SA_ID          |34329|14974                |
|TIER_2_NAME_PERSAL    |2390 |1174                 |
|TIER_3_EVENT_KEY      |1071 |1028                 |
+----------------------+-----+---------------------+



In [0]:
# ============================================================
# REPORTABLE RULE
# ============================================================
#
# WHAT THIS CELL DOES:
# Determines is_reportable for every row based on:
#   1. event_date falls inside Q4 2025
#   2. is_completed per source
#
# COMPLETION RULE PER SOURCE:
#   capturing_tool: attendance_status IN
#                   ('Attended', 'Partial attendance', 'Replacement')
#   chw:            any row counts as completed
#   online_export:  End Date IS NOT NULL (proxy)
#
# WHY online_end_dates.row_key MATCHES silver event_key EXACTLY:
# silver.participant_event.event_key is a SHA-256 of
#   source_system || source_file || source_sheet
#   || sa_id_normalised || surname || first_name
#   || course || event_date_raw
# The event_date_raw value for online_export comes from the
# Bronze column `Start Date` cast to string.
# This cell reconstructs the same hash so the join on
# event_key succeeds. If the formula diverges, online
# completions collapse to zero.
# ============================================================

# ------------------------------------------------------------
# Step 1: filter to reporting period
# ------------------------------------------------------------
in_period = fact_keyed.filter(
    (F.col("event_date") >= F.lit(REPORTING_PERIOD_START).cast("date")) &
    (F.col("event_date") <= F.lit(REPORTING_PERIOD_END).cast("date"))
)

# ------------------------------------------------------------
# Step 2: re-read online_export End Date from Bronze for the
# completion proxy. Join by event_key.
#
# The hash below MUST match the formula used in nb_03_Silver
# Cell 9 exactly:
#   source_system || source_file || source_sheet
#   || sa_id_normalised || surname || first_name
#   || course || event_date_raw
#
# For online_export rows:
#   source_system    = 'online_export' (literal)
#   source_file      = _source_file
#   source_sheet     = _source_sheet
#   sa_id_normalised = ID Number, stripped to digits
#   surname          = Last Name
#   first_name       = First Name
#   course           = Course Name
#   event_date_raw   = Start Date cast to string
# ------------------------------------------------------------
online_end_dates = (
    spark.table(f"{CATALOG}.bronze.online_export")
    .withColumn(
        "row_key",
        F.sha2(F.concat_ws("||",
            F.lit("online_export"),
            F.col("_source_file"),
            F.col("_source_sheet"),
            F.coalesce(
                F.regexp_replace(
                    F.regexp_replace(
                        F.trim(F.col("`ID Number`").cast("string")),
                        r"\.0$", ""
                    ),
                    r"[^0-9]", ""
                ),
                F.lit("")
            ),
            F.coalesce(F.col("`Last Name`"),   F.lit("")),
            F.coalesce(F.col("`First Name`"),  F.lit("")),
            F.coalesce(F.col("`Course Name`"), F.lit("")),
            F.coalesce(F.col("`Start Date`").cast("string"), F.lit("")),
        ), 256)
    )
    .select(
        F.col("row_key").alias("event_key"),
        F.col("`End Date`").alias("raw_end_date")
    )
    .dropDuplicates(["event_key"])
)

with_end_date = in_period.join(online_end_dates, "event_key", "left")

# ------------------------------------------------------------
# Step 3: apply the completion rule per source
# ------------------------------------------------------------
with_completion = with_end_date.withColumn(
    "is_completed",
    F.when(
        F.col("source_system") == "capturing_tool",
        F.col("attendance_status").isin("Attended", "Partial attendance", "Replacement")
    ).when(
        F.col("source_system") == "chw",
        F.lit(True)
    ).when(
        F.col("source_system") == "online_export",
        F.col("raw_end_date").isNotNull()
    ).otherwise(F.lit(False))
)

# ------------------------------------------------------------
# Step 4: mark reportable. Every row in period is reportable
# as an enrolment. Completion is a separate flag.
# ------------------------------------------------------------
reportable = with_completion.withColumn("is_reportable", F.lit(True))

if DEBUG:
    print("=" * 60)
    print("REPORTABLE ROWS IN Q4 2025")
    print("=" * 60)
    reportable.groupBy("source_system").agg(
        F.count("*").alias("enrolments"),
        F.sum(F.when(F.col("is_completed"), 1).otherwise(0)).alias("completions")
    ).orderBy("source_system").show()

    total = reportable.count()
    print(f"Total rows in Q4 2025: {total}")
    print(f"Total rows outside Q4 2025: {fact.count() - total}")

REPORTABLE ROWS IN Q4 2025
+--------------+----------+-----------+
| source_system|enrolments|completions|
+--------------+----------+-----------+
|capturing_tool|      2553|       2405|
| online_export|      2279|       1445|
+--------------+----------+-----------+

Total rows in Q4 2025: 4832
Total rows outside Q4 2025: 32958


In [0]:
# ============================================================
# WRITE DEDUP FACT TABLE AND EXCLUDED RECORDS
# ============================================================
#
# WHAT THIS CELL DOES:
# 1. Writes silver.participant_event_dedup with participant_key,
#    is_completed, is_reportable.
# 2. Writes silver.excluded_records for rows outside Q4 2025.
#
# WHY A LEFT ANTI JOIN INSTEAD OF SUBTRACT:
# DataFrame.subtract() is a distinct set difference. It compares
# all columns and collapses duplicates. Two rows with the same
# event_key but different other columns are not subtracted, and
# the resulting excluded_records contains event_keys that are
# also present in participant_event_dedup. A left anti join on
# event_key is exact and preserves row-level identity.
# ============================================================

# ------------------------------------------------------------
# Target table for dedup fact
# ------------------------------------------------------------
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.participant_event_dedup")

dedup_columns = reportable.select(
    F.col("event_key"),
    F.col("participant_key"),
    F.col("participant_match_tier"),
    F.col("source_system"),
    F.col("source_file"),
    F.col("source_sheet"),

    F.col("sa_id_normalised"),
    F.col("sa_id_luhn_valid"),
    F.col("persal_number"),
    F.col("professional_registration"),

    F.col("surname"),
    F.col("first_name"),
    F.col("email"),
    F.col("gender"),
    F.col("race"),
    F.col("disability"),

    F.col("employer_group_source"),
    F.col("employer_group_canonical"),

    F.col("profession_source"),
    F.col("profession_canonical"),
    F.col("professional_category"),

    F.col("facility_source"),
    F.col("facility_canonical"),
    F.col("facility_code"),
    F.col("facility_match_status"),

    F.col("district_source"),
    F.col("district_canonical"),
    F.col("sub_district"),

    F.col("course_source"),
    F.col("course_canonical"),
    F.col("course_code"),
    F.col("course_match_status"),

    F.col("attendance_status"),
    F.col("booking_status"),

    F.col("event_date"),
    F.col("reporting_period"),

    F.col("is_completed"),
    F.col("is_reportable"),
    F.lit(None).cast("string").alias("exclusion_reason"),

    F.col("ingested_at")
)

dedup_columns.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.participant_event_dedup")

# ------------------------------------------------------------
# Write excluded records using a LEFT ANTI JOIN on event_key.
# Every Silver fact row whose event_key does not appear in the
# dedup fact is excluded. Rows are preserved exactly.
# ------------------------------------------------------------
spark.sql(f"DELETE FROM {CATALOG}.{SILVER_SCHEMA}.excluded_records")

excluded = fact_keyed.join(
    reportable.select("event_key"),
    on="event_key",
    how="left_anti"
)

excluded.select(
    F.col("event_key"),
    F.col("source_system"),
    F.lit("OUTSIDE_REPORTING_PERIOD").alias("exclusion_reason"),
    F.concat_ws(" ", F.lit("event_date ="), F.col("event_date")).alias("details"),
    F.current_timestamp().alias("excluded_at"),
).write.format("delta").mode("append").saveAsTable(
    f"{CATALOG}.{SILVER_SCHEMA}.excluded_records"
)

if DEBUG:
    print("=" * 60)
    print("DEDUP FACT AND EXCLUDED RECORDS")
    print("=" * 60)

    dedup_count = spark.sql(
        f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.participant_event_dedup"
    ).collect()[0]["n"]

    excl_count = spark.sql(
        f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.excluded_records"
    ).collect()[0]["n"]

    silver_count = spark.sql(
        f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.participant_event"
    ).collect()[0]["n"]

    print(f"Dedup fact rows    : {dedup_count}")
    print(f"Excluded rows      : {excl_count}")
    print(f"Total accounted    : {dedup_count + excl_count}")
    print(f"Silver fact rows   : {silver_count}")
    print(f"Reconciliation     : {'PASS' if dedup_count + excl_count == silver_count else 'FAIL'}")
    print()
    print("Distinct participants in Q4 2025:")
    spark.sql(f"""
        SELECT COUNT(DISTINCT participant_key) AS unique_participants
        FROM {CATALOG}.{SILVER_SCHEMA}.participant_event_dedup
    """).show()

    print("Unique participants by source:")
    spark.sql(f"""
        SELECT
            source_system,
            COUNT(*) AS rows,
            COUNT(DISTINCT participant_key) AS unique_participants
        FROM {CATALOG}.{SILVER_SCHEMA}.participant_event_dedup
        GROUP BY source_system
        ORDER BY source_system
    """).show()

DEDUP FACT AND EXCLUDED RECORDS
Dedup fact rows    : 4832
Excluded rows      : 32958
Total accounted    : 37790
Silver fact rows   : 37790
Reconciliation     : PASS

Distinct participants in Q4 2025:
+-------------------+
|unique_participants|
+-------------------+
|               3259|
+-------------------+

Unique participants by source:
+--------------+----+-------------------+
| source_system|rows|unique_participants|
+--------------+----+-------------------+
|capturing_tool|2553|               1996|
| online_export|2279|               1408|
+--------------+----+-------------------+



In [0]:
# ============================================================
# FINALISE AUDIT
# ============================================================

end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

dedup_count = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.participant_event_dedup"
).collect()[0]["n"]

excl_count = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.excluded_records"
).collect()[0]["n"]

unique_p = spark.sql(
    f"SELECT COUNT(DISTINCT participant_key) AS n FROM {CATALOG}.{SILVER_SCHEMA}.participant_event_dedup"
).collect()[0]["n"]

spark.sql(f"DELETE FROM {CATALOG}.{AUDIT_SCHEMA}.data_quality_results WHERE run_id = '{run_id}'")

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.data_quality_results
    VALUES
    ('{run_id}', 'dedup_fact_row_count', 'completeness',
     'silver.participant_event_dedup', 'Rows in Q4 2025', '{dedup_count}',
     'PASS', NULL, current_timestamp()),
    ('{run_id}', 'dedup_excluded_rows', 'completeness',
     'silver.excluded_records', 'Rows outside Q4 2025', '{excl_count}',
     'PASS', NULL, current_timestamp()),
    ('{run_id}', 'dedup_unique_participants', 'uniqueness',
     'silver.participant_event_dedup', 'Distinct participants', '{unique_p}',
     'PASS', NULL, current_timestamp())
""")

spark.sql(f"""
    UPDATE {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    SET
        end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
        status           = 'SUCCESS',
        rows_in          = {fact_count if 'fact_count' in dir() else dedup_count + excl_count},
        rows_out         = {dedup_count},
        rows_rejected    = {excl_count},
        message          = 'Deduplication complete. Q4 2025 rows: {dedup_count}. Unique participants: {unique_p}. Excluded: {excl_count}.',
        duration_seconds = {duration}
    WHERE run_id = '{run_id}'
""")

if DEBUG:
    print()
    print("=" * 50)
    print("DEDUPLICATION SUMMARY")
    print("=" * 50)
    print(f"Run ID               : {run_id}")
    print(f"Dedup fact rows      : {dedup_count}")
    print(f"Unique participants  : {unique_p}")
    print(f"Excluded rows        : {excl_count}")
    print(f"Duration             : {duration}s")
    print(f"Status               : SUCCESS")
    print("=" * 50)

print("nb_04_Deduplication completed successfully.")


DEDUPLICATION SUMMARY
Run ID               : 70d8c26b-7ae8-4505-a8c1-d9a367f1dde3
Dedup fact rows      : 4832
Unique participants  : 3259
Excluded rows        : 32958
Duration             : 12s
Status               : SUCCESS
nb_04_Deduplication completed successfully.
